In [4]:
# lane_and_vehicle_system_clean.py
import cv2
import numpy as np
import time
import math
from ultralytics import YOLO

# ---------------- CONFIG ----------------
VIDEO_PATH = r"C:\Users\patel\Downloads\curved_lane.mp4"   # change to your local path if needed
OUTPUT_PATH = "output_lane_and_vehicle.mp4"
MODEL_PATH = "yolov8n.pt"
CONF_THRESHOLD = 0.35
DETECT_EVERY_N = 5 # run detector every N frames

TRACKER_TYPE = "CSRT"      # CSRT more stable
FPS_SMOOTH = 6             # smoothing window for speeds
PIXELS_TO_METERS_CONST = 25.0  # tune this depending on camera perspective
ALLOWED = {"car", "truck", "bus", "motorbike", "motorcycle"}
FORCE_CPU = True
# ----------------------------------------

lane_state = {
    "left_fit": None,    # np.array([A,B,C]) for x = A*y^2 + B*y + C
    "right_fit": None,
    "left_fit_history": [],   # store recent coeffs
    "right_fit_history": [],
    "max_history": 6
}

# ---------------- utilities ----------------
def create_tracker(name="CSRT"):
    return cv2.TrackerCSRT_create() if name == "CSRT" else cv2.TrackerKCF_create()

def euclid(a,b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def iou(boxA, boxB):
    xx1 = max(boxA[0], boxB[0]); yy1 = max(boxA[1], boxB[1])
    xx2 = min(boxA[2], boxB[2]); yy2 = min(boxA[3], boxB[3])
    interW = max(0, xx2 - xx1); interH = max(0, yy2 - yy1)
    interA = interW * interH
    if interA == 0:
        return 0.0
    aA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    bA = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    return interA / float(aA + bA - interA + 1e-9)

def calc_meters_from_pixels(pixel_distance, frame_h):
    PIXELS_TO_METERS = PIXELS_TO_METERS_CONST / frame_h
    return pixel_distance * PIXELS_TO_METERS

# ------- Lane detection helpers -------
def warp_perspective_for_lane(frame):
    h, w = frame.shape[:2]
    src = np.float32([
        [w * 0.10, h * 0.95],
        [w * 0.45, h * 0.62],
        [w * 0.55, h * 0.62],
        [w * 0.95, h * 0.95]
    ])
    dst = np.float32([
        [w * 0.2, h * 0.98],
        [w * 0.2, h * 0.02],
        [w * 0.8, h * 0.02],
        [w * 0.8, h * 0.98]
    ])
    M = cv2.getPerspectiveTransform(src, dst)
    Minv = cv2.getPerspectiveTransform(dst, src)
    warped = cv2.warpPerspective(frame, M, (w, h), flags=cv2.INTER_LINEAR)
    return warped, M, Minv

def binary_lane_mask(warped):
    """Combine color and gradient thresholds on warped image to get lane binary mask."""
    hls = cv2.cvtColor(warped, cv2.COLOR_BGR2HLS)
    L = hls[:,:,1]
    S = hls[:,:,2]
    sobelx = cv2.Sobel(L, cv2.CV_64F, 1, 0, ksize=3)
    abs_sobelx = np.absolute(sobelx)
    scaled = np.uint8(255 * abs_sobelx / (np.max(abs_sobelx) + 1e-6))
    sxbinary = np.zeros_like(scaled)
    sxbinary[(scaled >= 20) & (scaled <= 200)] = 1
    s_binary = np.zeros_like(S)
    s_binary[S > 120] = 1
    l_binary = np.zeros_like(L)
    l_binary[L > 200] = 1
    combined = np.zeros_like(sxbinary)
    combined[((s_binary == 1) | (l_binary == 1)) | (sxbinary == 1)] = 1
    return combined  # binary mask (0/1)

def sliding_window_polyfit(binary_warped, nwindows=9, margin=100, minpix=50):
    """Return left_fit, right_fit as polynomials (x = A*y^2 + B*y + C) in pixel coords of warped image."""
    h, w = binary_warped.shape
    histogram = np.sum(binary_warped[h//2:,:], axis=0)
    midpoint = int(histogram.shape[0] / 2)
    leftx_base = np.argmax(histogram[:midpoint])
    rightx_base = np.argmax(histogram[midpoint:]) + midpoint

    nonzero = binary_warped.nonzero()
    nonzeroy = np.array(nonzero[0])
    nonzerox = np.array(nonzero[1])

    leftx_current = leftx_base
    rightx_current = rightx_base

    window_height = int(h / nwindows)
    left_lane_inds = []
    right_lane_inds = []

    for window in range(nwindows):
        win_y_low = h - (window + 1) * window_height
        win_y_high = h - window * window_height
        win_xleft_low = leftx_current - margin
        win_xleft_high = leftx_current + margin
        win_xright_low = rightx_current - margin
        win_xright_high = rightx_current + margin

        good_left_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) &
                          (nonzerox >= win_xleft_low) & (nonzerox < win_xleft_high)).nonzero()[0]
        good_right_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) &
                           (nonzerox >= win_xright_low) & (nonzerox < win_xright_high)).nonzero()[0]

        left_lane_inds.append(good_left_inds)
        right_lane_inds.append(good_right_inds)

        if len(good_left_inds) > minpix:
            leftx_current = int(np.mean(nonzerox[good_left_inds]))
        if len(good_right_inds) > minpix:
            rightx_current = int(np.mean(nonzerox[good_right_inds]))

    left_lane_inds = np.concatenate(left_lane_inds) if len(left_lane_inds) > 0 else np.array([], dtype=np.int32)
    right_lane_inds = np.concatenate(right_lane_inds) if len(right_lane_inds) > 0 else np.array([], dtype=np.int32)

    left_fit = None
    right_fit = None
    if left_lane_inds.size > 0:
        leftx = nonzerox[left_lane_inds]; lefty = nonzeroy[left_lane_inds]
        if lefty.size >= 3:
            left_fit = np.polyfit(lefty, leftx, 2)  # x = f(y)
    if right_lane_inds.size > 0:
        rightx = nonzerox[right_lane_inds]; righty = nonzeroy[right_lane_inds]
        if righty.size >= 3:
            right_fit = np.polyfit(righty, rightx, 2)
    return left_fit, right_fit

def smooth_fit(lane_state, new_left, new_right):
    """Exponential/rolling smoothing of polynomial coeffs stored in lane_state global."""
    def update_and_avg(history_list, new_coeff):
        if new_coeff is None:
            if len(history_list) == 0:
                return None
            arr = np.array(history_list)
            return np.mean(arr, axis=0)
        history_list.append(new_coeff)
        if len(history_list) > lane_state["max_history"]:
            history_list.pop(0)
        return np.mean(np.array(history_list), axis=0)

    avg_left = update_and_avg(lane_state["left_fit_history"], new_left)
    avg_right = update_and_avg(lane_state["right_fit_history"], new_right)

    lane_state["left_fit"] = avg_left
    lane_state["right_fit"] = avg_right
    return avg_left, avg_right

def poly_points_from_fit(fit, y_vals):
    if fit is None:
        return None
    A,B,C = fit
    x_vals = A * (y_vals**2) + B * y_vals + C
    return x_vals.astype(np.int32)

def get_lane_lines(frame, lane_state):
    """Warp -> binary -> sliding window -> smooth -> return points for drawing and curvature (meters)."""
    h, w = frame.shape[:2]
    warped, M, Minv = warp_perspective_for_lane(frame)
    binary = binary_lane_mask(warped)
    left_fit_new, right_fit_new = sliding_window_polyfit(binary)
    left_avg, right_avg = smooth_fit(lane_state, left_fit_new, right_fit_new)

    ploty = np.linspace(0, h-1, h)
    leftx = poly_points_from_fit(left_avg, ploty) if left_avg is not None else None
    rightx = poly_points_from_fit(right_avg, ploty) if right_avg is not None else None

    left_pts = []
    right_pts = []
    if leftx is not None:
        left_pts = [(int(leftx[i]), int(ploty[i])) for i in range(0, len(ploty), max(1, h//120))]
        left_pts = np.array(left_pts, dtype=np.float32).reshape(-1,1,2)
        left_pts = cv2.perspectiveTransform(left_pts, Minv).reshape(-1,2).astype(int).tolist()
    if rightx is not None:
        right_pts = [(int(rightx[i]), int(ploty[i])) for i in range(0, len(ploty), max(1, h//120))]
        right_pts = np.array(right_pts, dtype=np.float32).reshape(-1,1,2)
        right_pts = cv2.perspectiveTransform(right_pts, Minv).reshape(-1,2).astype(int).tolist()

    # curvature estimate in meters via finite differences on meter-scaled points
    def curvature_from_fit(fit, frame_h):
        if fit is None:
            return None
        Apx, Bpx, Cpx = fit
        s = PIXELS_TO_METERS_CONST / frame_h
        ys = np.linspace(0, frame_h-1, 100) * s
        xs = (Apx * (ys/s)**2 + Bpx * (ys/s) + Cpx) * s
        a,b,c = np.polyfit(ys, xs, 2)
        y_eval = ys[-1]
        dxdy = 2*a*y_eval + b
        d2 = 2*a
        if abs(d2) < 1e-9:
            return None
        R = (1 + dxdy**2)**1.5 / abs(d2)
        return abs(R)

    curv_left = curvature_from_fit(left_avg, h)
    curv_right = curvature_from_fit(right_avg, h)
    curv = None
    if curv_left is not None and curv_right is not None:
        curv = (curv_left + curv_right) / 2.0
    elif curv_left is not None:
        curv = curv_left
    elif curv_right is not None:
        curv = curv_right

    return left_pts, right_pts, curv

# ---------------- MAIN ----------------
def main():
    print("Loading YOLO model...")
    model = YOLO(MODEL_PATH)
    if FORCE_CPU:
        model.to('cpu')
    model.overrides['verbose'] = False

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print("Cannot open video:", VIDEO_PATH)
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video {frame_w}x{frame_h} @ {fps:.2f} FPS")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (frame_w, frame_h))

    trackers = {}      # id -> {'tracker','bbox','positions':[((x,y),frame_idx)], 'speed_kmh','class','last_seen'}
    speed_hist = {}    # id -> [speeds]
    next_id = 0
    frame_idx = 0

    print("Starting processing. press 'q' to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        t0 = time.time()

        detections = []
        if frame_idx % DETECT_EVERY_N == 0:
            small = cv2.resize(frame, (640,640))
            res = model(small, imgsz=640, conf=CONF_THRESHOLD, verbose=False)[0]
            if hasattr(res, "boxes") and len(res.boxes) > 0:
                boxes = res.boxes.xyxy.cpu().numpy()
                classes = res.boxes.cls.cpu().numpy().astype(int)
                confs = res.boxes.conf.cpu().numpy()
                sx = frame.shape[1] / 640.0
                sy = frame.shape[0] / 640.0
                for b,c,cf in zip(boxes, classes, confs):
                    name = model.names[int(c)]
                    if name == "motorcycle":
                        name = "motorbike"
                    if name in ALLOWED:
                        x1,y1,x2,y2 = b[:4]
                        x1 = int(x1 * sx); x2 = int(x2 * sx)
                        y1 = int(y1 * sy); y2 = int(y2 * sy)
                        detections.append((x1,y1,x2,y2,name,float(cf)))

        # update trackers each frame and remove lost
        lost = []
        for tid, info in list(trackers.items()):
            ok, box = info['tracker'].update(frame)
            if ok:
                x,y,w_box,h_box = box
                x1 = int(x); y1 = int(y); x2 = int(x + w_box); y2 = int(y + h_box)
                trackers[tid]['bbox'] = (x1,y1,x2,y2)
                trackers[tid]['last_seen'] = frame_idx
            else:
                if frame_idx - info.get('last_seen', frame_idx) > int(fps * 1.0):
                    lost.append(tid)
        for rid in lost:
            trackers.pop(rid, None)
            speed_hist.pop(rid, None)

        # match detection->trackers
        unmatched = []
        for det in detections:
            dx1,dy1,dx2,dy2,dname,dconf = det
            best_iou = 0.0; best_tid = None
            for tid, info in trackers.items():
                val = iou((dx1,dy1,dx2,dy2), info['bbox'])
                if val > best_iou:
                    best_iou = val; best_tid = tid
            if best_iou >= 0.35 and best_tid is not None:
                try:
                    new_tr = create_tracker(TRACKER_TYPE)
                    new_tr.init(frame, (dx1,dy1,dx2-dx1,dy2-dy1))
                    trackers[best_tid]['tracker'] = new_tr
                    trackers[best_tid]['bbox'] = (dx1,dy1,dx2,dy2)
                    trackers[best_tid]['class'] = dname
                    trackers[best_tid]['last_seen'] = frame_idx
                except Exception:
                    pass
            else:
                unmatched.append(det)

        for det in unmatched:
            dx1,dy1,dx2,dy2,dname,dconf = det
            try:
                tr = create_tracker(TRACKER_TYPE)
                tr.init(frame, (dx1,dy1,dx2-dx1,dy2-dy1))
                trackers[next_id] = {'tracker':tr, 'bbox':(dx1,dy1,dx2,dy2), 'positions':[], 'speed_kmh':0.0, 'class':dname, 'last_seen':frame_idx}
                next_id += 1
            except Exception:
                pass

        # get lane lines + curvature
        left_pts, right_pts, curv = get_lane_lines(frame, lane_state)

        # update trackers positions & speeds (from bottom-center)
        for tid, info in trackers.items():
            x1,y1,x2,y2 = info['bbox']
            if x2 <= x1 or y2 <= y1:
                continue
            bc = ((x1 + x2)//2, y2)
            info['positions'].append((bc, frame_idx))
            if len(info['positions']) > FPS_SMOOTH:
                info['positions'].pop(0)
            if len(info['positions']) >= 2:
                (x_prev, y_prev), f_prev = info['positions'][-2]
                (x_cur, y_cur), f_cur = info['positions'][-1]
                pixel_dist = euclid((x_prev,y_prev),(x_cur,y_cur))
                meters = calc_meters_from_pixels(pixel_dist, frame_h)
                dt_seconds = max(1e-6, (f_cur - f_prev) * (1.0 / fps))
                speed_kmh_new = (meters / dt_seconds) * 3.6
                prev = info.get('speed_kmh', speed_kmh_new)
                info['speed_kmh'] = 0.75 * prev + 0.25 * speed_kmh_new
                if tid not in speed_hist:
                    speed_hist[tid] = []
                speed_hist[tid].append(info['speed_kmh'])
                if len(speed_hist[tid]) > FPS_SMOOTH:
                    speed_hist[tid].pop(0)
                display_speed = float(np.mean(speed_hist[tid]))
            else:
                display_speed = 0.0
            info['display_speed'] = display_speed

        # ------- draw final overlay -------
        vis = frame.copy()

        # Draw lane area (polygon)
        if len(left_pts) > 2 and len(right_pts) > 2:
            lane_polygon = np.array(left_pts + right_pts[::-1], dtype=np.int32)
            cv2.fillPoly(vis, [lane_polygon], (0, 200, 0, 40))

        # Draw left and right lane curves
        if len(left_pts) >= 2:
            cv2.polylines(vis, [np.array(left_pts, np.int32)], False, (0, 255, 255), 5)
        if len(right_pts) >= 2:
            cv2.polylines(vis, [np.array(right_pts, np.int32)], False, (0, 255, 255), 5)

        # Draw trackers
        for tid, info in trackers.items():
            x1,y1,x2,y2 = info['bbox']
            clsname = info.get('class','obj')
            spd = info.get('display_speed', 0.0)
            cval = int(min(255, spd * 3))
            color = (0, 255 - cval, cval)
            cv2.rectangle(vis, (x1,y1), (x2,y2), color, 2)
            cv2.putText(vis, f"ID:{tid} {clsname}", (x1, y1-18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
            cv2.putText(vis, f"{spd:.1f} km/h", (x1, y2+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
            pts = [p[0] for p in info['positions']]
            if len(pts) >= 2:
                cv2.polylines(vis, [np.array(pts, dtype=np.int32)], False, (255,255,0), 2)
            bc = ((x1+x2)//2, y2)
            cv2.circle(vis, bc, 3, (255,255,0), -1)

        # ---------- Top-left display: only curvature + direction ----------
        if curv is not None:
            display_curv_text = f"Curvature: {curv:.1f} m"
        else:
            display_curv_text = "Curvature: N/A"

        direction = "N/A"
        if len(left_pts) >= 1 and len(right_pts) >= 1:
            left_bottom_x = left_pts[-1][0]
            right_bottom_x = right_pts[-1][0]
            lane_center_x = (left_bottom_x + right_bottom_x) / 2.0
            img_center_x = vis.shape[1] / 2.0
            tol = vis.shape[1] * 0.03  # 3% tolerance

            if lane_center_x - img_center_x > tol:
                direction = "Right"
            elif lane_center_x - img_center_x < -tol:
                direction = "Left"
            else:
                direction = "Straight"

        display_dir_text = f"Direction: {direction}"
        cv2.putText(vis, display_curv_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (240,240,240), 2, cv2.LINE_AA)
        cv2.putText(vis, display_dir_text,  (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (240,240,240), 2, cv2.LINE_AA)
        # ---------------------------------------------------------------

        out.write(vis)
        cv2.imshow("Lane + Vehicle System", vis)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Saved:", OUTPUT_PATH)

if __name__ == "__main__":
    main()


Loading YOLO model...
Video 1280x720 @ 25.00 FPS
Starting processing. press 'q' to quit.
Saved: output_lane_and_vehicle.mp4
